# 技能4 · Day 3 上机：Agent经济 + 新兴商业模式

**版本**：v5.0 学习材料包
**配套**：notes.md（讲义）｜ data/README.md（真实库/数据）｜ solution.ipynb（参考答案，做完再看）

## 学习目标
学完你能：
1. 用 **mesa** 构建Agent经济仿真--消费者Agent/商家Agent/AI中介Agent三类主体交互，涌现市场价格/财富分布/存活率
2. 理解Agent经济三层模型（Agent-as-Tool / Agent-as-Worker / Agent-as-Actor）的计算化建模方法
3. 用 **pandas + matplotlib** 分析仿真涌现结果--基尼系数/价格分布/Agent存活率/A2A交易量
4. 理解推理成本对Agent经济行为的约束，以及平台抽成（30%真实比例）对生态的影响
5. 建立天道推演×多Agent仿真的同构认知--仿真本质是计算化的天道推演沙盘

## 真实库与真实数据
- **mesa**（agent-based modeling 框架）：构建Agent经济仿真
- **pandas + matplotlib**：仿真结果分析与可视化
- **numpy**：仿真数学计算
- **真实经济参数**：平台抽成30%（Apple/Amazon真实比例）、Token定价$5/1M（GPT-4o真实定价）、推理成本约束

> 详见 data/README.md


## 0. 环境准备

首次运行需安装依赖（取消注释执行一次）：

> 所有库（mesa/pandas/matplotlib/numpy）均为本地可用库，不需要API Key。


In [ ]:
# !pip install mesa pandas matplotlib numpy -q

import warnings
warnings.filterwarnings('ignore')

import mesa
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # 非交互式后端
import matplotlib.pyplot as plt
from mesa.datacollection import DataCollector

print(f"mesa {mesa.__version__} | pandas {pd.__version__} | numpy {np.__version__}")
print("Agent经济仿真环境就绪")


## 1. 真实经济参数

Agent经济仿真的参数基于真实世界的经济数据，不是编造的数字：

| 参数 | 值 | 真实来源 |
|------|-----|---------|
| 平台抽成率 | 30% | Apple App Store / Amazon Marketplace 真实抽成比例 |
| Token定价 | $5/1M tokens | GPT-4o input 定价（OpenAI 2024-2025定价页） |
| 每次匹配推理token | 500 tokens | Agent协商/比价/决策的合理token消耗 |
| 推理成本/匹配 | ~$0.0025 | 500 tokens × $5/1M |

**推理成本是Agent经济的核心约束**--AI中介每次匹配都消耗token，这直接影响其经济可行性。


In [ ]:
# 真实经济参数（可追溯来源）
# 平台抽成率 30%: Apple App Store / Amazon Marketplace 真实比例
PLATFORM_COMMISSION_RATE = 0.30
# Token定价: GPT-4o input ~$5/1M tokens (OpenAI 2024-2025定价页)
TOKEN_PRICE_PER_1M = 5.0
# 每次匹配推理token消耗（Agent协商/比价/决策）
TOKENS_PER_MATCH = 500
# 每次匹配的推理成本
REASONING_COST_PER_MATCH = (TOKENS_PER_MATCH / 1_000_000) * TOKEN_PRICE_PER_1M

print(f"平台抽成率: {PLATFORM_COMMISSION_RATE*100:.0f}%")
print(f"推理成本/匹配: ${REASONING_COST_PER_MATCH:.4f} ({TOKENS_PER_MATCH} tokens × ${TOKEN_PRICE_PER_1M}/1M)")
print(f"  -> 这是AI中介每次匹配的净利润约束")


## 2. TODO 1：消费者Agent

**消费者Agent** 是Agent经济中的需求方。每个消费者有：
- `wealth`：预算（初始1000）
- `demand`：需求量
- `alive`：是否存活（预算耗尽则破产）
- `purchases`：累计购买次数

**行为逻辑**：
1. 如果有AI中介可用，通过中介寻找最低价商家购买（支付 中介费）
2. 如果无中介，随机选择商家直接购买
3. 预算耗尽（< $1）则破产

**营销映射**：消费者Agent代客比价是Agent经济在营销中的典型实例。


In [ ]:
# TODO 1：消费者Agent
# 提示：继承mesa.Agent，实现step()方法
#   属性：wealth=1000, demand=1, alive=True, purchases=0
#   行为：通过AI中介找最低价 / 直接随机购买；预算<1则破产

class ConsumerAgent(mesa.Agent):
    """消费者Agent：有预算，通过AI中介寻找最优价格购买商品。"""
    def __init__(self, model, initial_budget=1000.0, demand=1):
        super().__init__(model)
        self.wealth = initial_budget
        self.demand = demand
        self.alive = True
        self.purchases = 0
        self.total_spent = 0.0

    def step(self):
        if not self.alive:
            return
        # 找存活且有库存的商家
        merchants = [a for a in self.model.agents
                     if isinstance(a, MerchantAgent) and a.alive and a.inventory > 0]
        if not merchants:
            return
        # 找可用的AI中介
        intermediaries = [a for a in self.model.agents
                          if isinstance(a, AIIntermediaryAgent) and a.alive]

        if intermediaries:
            # 通过中介找最低价
            intermediary = self.random.choice(intermediaries)
            best_merchant = min(merchants, key=lambda m: m.price)
            total_cost = best_merchant.price + intermediary.fee
            if self.wealth >= total_cost:
                self.wealth -= total_cost
                self.total_spent += total_cost
                best_merchant.sell_via_intermediary(intermediary)
                self.purchases += 1
        else:
            # 直接购买（随机商家）
            merchant = self.random.choice(merchants)
            if self.wealth >= merchant.price:
                self.wealth -= merchant.price
                self.total_spent += merchant.price
                merchant.sell_direct()
                self.purchases += 1

        # 破产检查
        if self.wealth < 1.0:
            self.alive = False

# 验证
print(f"ConsumerAgent 定义完成")
print(f"  推理: 消费者通过AI中介比价 = Agent经济中的'代客比价'")


## 3. TODO 2：商家Agent

**商家Agent** 是Agent经济中的供给方。每个商家有：
- `wealth`：资金（初始500）
- `base_cost`：生产成本
- `price`：当前售价（动态调整）
- `inventory`：库存
- `commission_paid`：累计支付平台抽成

**行为逻辑**：
1. 通过中介或直接销售，每次销售支付30%平台抽成
2. 库存高则降价，库存低则涨价（动态定价）
3. 定期补货（消耗资金）
4. 资金为负则破产

**真实参数**：平台抽成30%来自Apple App Store / Amazon Marketplace的真实比例。


In [ ]:
# TODO 2：商家Agent
# 提示：继承mesa.Agent
#   属性：wealth=500, base_cost=10, price=20, inventory=100, commission_paid=0
#   方法：sell_via_intermediary(int) / sell_direct() -> 支付30%抽成 + 动态调价
#   行为：补货 + 破产检查

class MerchantAgent(mesa.Agent):
    """商家Agent：定价、生产、支付平台抽成。"""
    def __init__(self, model, initial_wealth=500.0, base_cost=10.0):
        super().__init__(model)
        self.wealth = initial_wealth
        self.base_cost = base_cost
        self.price = base_cost * 2.0  # 初始2倍加价
        self.inventory = 100
        self.alive = True
        self.sales = 0
        self.commission_paid = 0.0

    def sell_via_intermediary(self, intermediary):
        revenue = self.price
        commission = revenue * PLATFORM_COMMISSION_RATE  # 30%平台抽成
        self.wealth += revenue - commission
        self.commission_paid += commission
        self.inventory -= 1
        self.sales += 1
        intermediary.process_transaction()
        self._adjust_price()

    def sell_direct(self):
        revenue = self.price
        commission = revenue * PLATFORM_COMMISSION_RATE
        self.wealth += revenue - commission
        self.commission_paid += commission
        self.inventory -= 1
        self.sales += 1
        self._adjust_price()

    def _adjust_price(self):
        # 库存高降价，库存低涨价
        if self.inventory > 60:
            self.price = max(self.base_cost, self.price * 0.97)
        elif self.inventory < 30:
            self.price = self.price * 1.03

    def step(self):
        if not self.alive:
            return
        # 补货
        restock_cost = max(0, (100 - self.inventory)) * self.base_cost * 0.5
        if self.wealth >= restock_cost and self.inventory < 50:
            self.wealth -= restock_cost
            self.inventory = 100
        if self.wealth < 0:
            self.alive = False

print(f"MerchantAgent 定义完成")
print(f"  真实参数: 平台抽成 {PLATFORM_COMMISSION_RATE*100:.0f}% (Apple/Amazon真实比例)")


## 4. TODO 3：AI中介Agent

**AI中介Agent** 是Agent经济的核心创新--它用AI能力匹配供需，每次匹配消耗推理token。

核心属性：
- `wealth`：资金（初始200）
- `fee`：中介费（动态调整）
- `transactions`：累计匹配次数
- `a2a_trades`：A2A交易次数（与其他中介的交易）
- `total_reasoning_cost`：累计推理成本

**行为逻辑**：
1. 每次匹配收取fee，支付推理成本（500 tokens × $5/1M = $0.0025）
2. 以15%概率与其他中介进行A2A信息交换（支付小额费用）
3. 根据竞争者平均费率动态调整自己的fee
4. 资金为负则破产

**A2A经济**：Agent-to-Agent交易是Agent经济最前沿的形态--Agent间自主协商、交易。


In [ ]:
# TODO 3：AI中介Agent
# 提示：继承mesa.Agent
#   属性：wealth=200, fee=2.0, transactions=0, a2a_trades=0, total_reasoning_cost=0
#   方法：process_transaction() -> 收fee, 支付推理成本
#   行为：A2A交易(15%概率) + 动态调费 + 破产检查

class AIIntermediaryAgent(mesa.Agent):
    """AI中介Agent：匹配供需、收取费用、支付推理成本、进行A2A交易。"""
    def __init__(self, model, initial_wealth=200.0, fee=2.0):
        super().__init__(model)
        self.wealth = initial_wealth
        self.fee = fee
        self.alive = True
        self.transactions = 0
        self.a2a_trades = 0
        self.total_reasoning_cost = 0.0

    def process_transaction(self):
        # 收取中介费，支付推理成本
        self.wealth += self.fee
        self.wealth -= REASONING_COST_PER_MATCH
        self.total_reasoning_cost += REASONING_COST_PER_MATCH
        self.transactions += 1

    def step(self):
        if not self.alive:
            return
        # A2A: 与其他中介交换市场信息（Agent-to-Agent经济）
        others = [a for a in self.model.agents
                  if isinstance(a, AIIntermediaryAgent) and a != self and a.alive]
        if others and self.random.random() < 0.15:
            partner = self.random.choice(others)
            a2a_fee = 0.5
            if self.wealth >= a2a_fee:
                self.wealth -= a2a_fee
                partner.wealth += a2a_fee
                self.a2a_trades += 1
                partner.a2a_trades += 1

        # 动态调费（基于竞争者平均费率）
        if others:
            avg_fee = sum(o.fee for o in others) / len(others)
            if self.fee > avg_fee:
                self.fee = max(0.5, self.fee * 0.95)
            else:
                self.fee = min(5.0, self.fee * 1.05)

        if self.wealth < 0:
            self.alive = False

print(f"AIIntermediaryAgent 定义完成")
print(f"  推理成本/匹配: ${REASONING_COST_PER_MATCH:.4f}")
print(f"  A2A经济: 中介间自主信息交换")


## 5. TODO 4：Agent经济模型 + DataCollector

**AgentEconomyModel** 整合三类Agent，用mesa的DataCollector追踪涌现指标：

| 指标 | 含义 |
|------|------|
| `gini` | 基尼系数（财富不平等程度） |
| `avg_price` | 市场平均价格 |
| `price_std` | 价格标准差（价格收敛程度） |
| `n_alive_*` | 各类Agent存活数 |
| `total_a2a_trades` | 累计A2A交易量 |
| `total_commission` | 累计平台抽成 |

**天道推演映射**：模型每一步step()就是一次沙盘推演--观察不同Agent策略下的经济涌现。


In [ ]:
# TODO 4：Agent经济模型 + DataCollector
# 提示：继承mesa.Model
#   __init__: 创建三类Agent + DataCollector(8个model_reporters + 3个agent_reporters)
#   _compute_gini: 基尼系数公式 G = 2*sum((i+1)*w)/(n*sum(w)) - (n+1)/n
#   step: agents.shuffle_do("step") + datacollector.collect

class AgentEconomyModel(mesa.Model):
    """Agent经济仿真模型：消费者/商家/AI中介交互，涌现市场价格/财富分布/存活率。"""
    def __init__(self, n_consumers=50, n_merchants=10, n_intermediaries=3, seed=42):
        super().__init__(rng=seed)
        # 创建Agent
        ConsumerAgent.create_agents(model=self, n=n_consumers,
                                     initial_budget=1000.0, demand=1)
        MerchantAgent.create_agents(model=self, n=n_merchants,
                                     initial_wealth=500.0, base_cost=10.0)
        AIIntermediaryAgent.create_agents(model=self, n=n_intermediaries,
                                           initial_wealth=200.0, fee=2.0)

        self.datacollector = DataCollector(
            model_reporters={
                "gini": self._compute_gini,
                "avg_price": self._compute_avg_price,
                "price_std": self._compute_price_std,
                "n_alive_consumers": lambda m: sum(1 for a in m.agents
                    if isinstance(a, ConsumerAgent) and a.alive),
                "n_alive_merchants": lambda m: sum(1 for a in m.agents
                    if isinstance(a, MerchantAgent) and a.alive),
                "n_alive_intermediaries": lambda m: sum(1 for a in m.agents
                    if isinstance(a, AIIntermediaryAgent) and a.alive),
                "total_a2a_trades": lambda m: sum(a.a2a_trades for a in m.agents
                    if isinstance(a, AIIntermediaryAgent)),
                "total_intermediary_transactions": lambda m: sum(a.transactions for a in m.agents
                    if isinstance(a, AIIntermediaryAgent)),
                "total_commission": lambda m: sum(a.commission_paid for a in m.agents
                    if isinstance(a, MerchantAgent)),
            },
            agent_reporters={
                "wealth": "wealth",
                "agent_type": lambda a: type(a).__name__,
                "alive": "alive",
            }
        )
        self.datacollector.collect(self)

    def _compute_gini(self):
        wealths = sorted([a.wealth for a in self.agents if a.wealth > 0])
        n = len(wealths)
        if n == 0 or sum(wealths) == 0:
            return 0.0
        cum = sum((i + 1) * w for i, w in enumerate(wealths))
        return (2 * cum) / (n * sum(wealths)) - (n + 1) / n

    def _compute_avg_price(self):
        prices = [a.price for a in self.agents
                  if isinstance(a, MerchantAgent) and a.alive]
        return float(np.mean(prices)) if prices else 0.0

    def _compute_price_std(self):
        prices = [a.price for a in self.agents
                  if isinstance(a, MerchantAgent) and a.alive]
        return float(np.std(prices)) if prices else 0.0

    def step(self):
        self.agents.shuffle_do("step")
        self.datacollector.collect(self)

# 验证
model = AgentEconomyModel(n_consumers=50, n_merchants=10, n_intermediaries=3, seed=42)
print(f"模型创建成功: {len(model.agents)} agents")
print(f"  消费者: {sum(1 for a in model.agents if isinstance(a, ConsumerAgent))}")
print(f"  商家: {sum(1 for a in model.agents if isinstance(a, MerchantAgent))}")
print(f"  AI中介: {sum(1 for a in model.agents if isinstance(a, AIIntermediaryAgent))}")
print(f"  初始基尼: {model._compute_gini():.4f}")


## 6. TODO 5：运行仿真 + 提取数据

运行Agent经济仿真100个tick，用DataCollector提取时间序列数据到pandas DataFrame。

**关键问题**：
- 基尼系数如何变化？（财富是否越来越集中？）
- 市场价格是否收敛？
- 哪类Agent最先破产？
- A2A交易量增长趋势如何？


In [ ]:
# TODO 5：运行仿真 + 提取数据
# 提示：运行100步，用datacollector提取model_vars和agent_vars到DataFrame
#   打印仿真规模、最终基尼、价格分布、存活数、A2A交易量

model = AgentEconomyModel(n_consumers=50, n_merchants=10, n_intermediaries=3, seed=42)
N_STEPS = 100
for i in range(N_STEPS):
    model.step()

# 提取数据到pandas DataFrame
model_df = model.datacollector.get_model_vars_dataframe()
agent_df = model.datacollector.get_agent_vars_dataframe()

print(f"仿真规模: {N_STEPS} ticks, {len(model.agents)} agents")
print(f"\n--- 最终状态 (tick {N_STEPS}) ---")
print(f"基尼系数: {model_df['gini'].iloc[-1]:.4f}")
print(f"平均价格: ${model_df['avg_price'].iloc[-1]:.2f}")
print(f"价格标准差: ${model_df['price_std'].iloc[-1]:.2f}")
print(f"存活消费者: {int(model_df['n_alive_consumers'].iloc[-1])}/50")
print(f"存活商家: {int(model_df['n_alive_merchants'].iloc[-1])}/10")
print(f"存活AI中介: {int(model_df['n_alive_intermediaries'].iloc[-1])}/3")
print(f"累计A2A交易: {int(model_df['total_a2a_trades'].iloc[-1])}")
print(f"累计中介交易: {int(model_df['total_intermediary_transactions'].iloc[-1])}")
print(f"累计平台抽成: ${model_df['total_commission'].iloc[-1]:.2f}")

print(f"\n--- 基尼系数变化 ---")
print(f"初始: {model_df['gini'].iloc[0]:.4f}")
print(f"第50步: {model_df['gini'].iloc[50]:.4f}")
print(f"最终: {model_df['gini'].iloc[-1]:.4f}")

print(f"\n--- 价格分布 (最终tick) ---")
final_prices = sorted([a.price for a in model.agents
                       if isinstance(a, MerchantAgent) and a.alive])
print(f"价格列表: {[round(p,2) for p in final_prices]}")
print(f"价格区间: ${min(final_prices):.2f} - ${max(final_prices):.2f}")

print(f"\nmodel_df 形状: {model_df.shape}")
print(f"agent_df 形状: {agent_df.shape}")


## 7. TODO 6：仿真结果分析与可视化

用pandas分析仿真涌现结果，用matplotlib绘制4个子图：
1. 基尼系数随时间变化（财富不平等趋势）
2. 市场价格分布（均值±标准差）
3. Agent存活数（消费者/商家/中介三条线）
4. A2A交易 vs 中介交易量对比

**涌现分析**：这些指标是Agent个体行为的涌现结果--没有任何单个Agent"知道"全局价格或基尼系数，它们在交互中自然产生。


In [ ]:
# TODO 6：仿真结果分析与可视化
# 提示：用matplotlib绘制4个子图
#   1. 基尼系数随时间变化 2. 市场价格分布(均值±std)
#   3. Agent存活数(3条线) 4. A2A vs 中介交易量

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# 子图1: 基尼系数
axes[0, 0].plot(model_df.index, model_df['gini'], color='#2563eb', linewidth=1.5)
axes[0, 0].set_title('基尼系数 (财富不平等)', fontsize=12)
axes[0, 0].set_xlabel('Tick')
axes[0, 0].set_ylabel('Gini')
axes[0, 0].axhline(y=0.3, color='r', linestyle='--', alpha=0.5, label='0.3 警戒线')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 子图2: 市场价格分布
axes[0, 1].plot(model_df.index, model_df['avg_price'], color='#16a34a',
                linewidth=1.5, label='平均价格')
axes[0, 1].fill_between(model_df.index,
    model_df['avg_price'] - model_df['price_std'],
    model_df['avg_price'] + model_df['price_std'],
    alpha=0.2, color='#16a34a', label='±1 std')
axes[0, 1].set_title('市场价格分布', fontsize=12)
axes[0, 1].set_xlabel('Tick')
axes[0, 1].set_ylabel('Price ($)')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# 子图3: Agent存活数
axes[1, 0].plot(model_df.index, model_df['n_alive_consumers'],
                label='消费者', color='#2563eb', linewidth=1.5)
axes[1, 0].plot(model_df.index, model_df['n_alive_merchants'],
                label='商家', color='#dc2626', linewidth=1.5)
axes[1, 0].plot(model_df.index, model_df['n_alive_intermediaries'],
                label='AI中介', color='#9333ea', linewidth=1.5)
axes[1, 0].set_title('Agent存活数', fontsize=12)
axes[1, 0].set_xlabel('Tick')
axes[1, 0].set_ylabel('Count')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# 子图4: A2A vs 中介交易量
axes[1, 1].plot(model_df.index, model_df['total_a2a_trades'],
                label='A2A交易', color='#ea580c', linewidth=1.5)
axes[1, 1].plot(model_df.index, model_df['total_intermediary_transactions'],
                label='中介匹配交易', color='#0891b2', linewidth=1.5)
axes[1, 1].set_title('A2A vs 中介交易量', fontsize=12)
axes[1, 1].set_xlabel('Tick')
axes[1, 1].set_ylabel('Cumulative')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle('Agent经济仿真涌现结果 (mesa)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('agent_economy_simulation.png', dpi=100, bbox_inches='tight')
plt.show()
print("图表已保存: agent_economy_simulation.png")

# 涌现分析
print("\n=== 涌现分析 ===")
print(f"1. 财富不平等: 基尼系数从 {model_df['gini'].iloc[0]:.4f} 升至 {model_df['gini'].iloc[-1]:.4f}")
print(f"   -> AI中介作为信息中介，抽取了市场信息租金，导致财富集中")
print(f"2. 价格收敛: 平均价格 ${model_df['avg_price'].iloc[0]:.2f} -> ${model_df['avg_price'].iloc[-1]:.2f}")
print(f"   -> 中介比价促进价格发现，但未完全收敛（信息摩擦）")
print(f"3. Agent存活: 消费者 {int(model_df['n_alive_consumers'].iloc[-1])}/50, "
      f"商家 {int(model_df['n_alive_merchants'].iloc[-1])}/10, "
      f"中介 {int(model_df['n_alive_intermediaries'].iloc[-1])}/3")
print(f"   -> 消费者预算耗尽是最主要的退出机制")
print(f"4. A2A交易: {int(model_df['total_a2a_trades'].iloc[-1])} 次")
print(f"   -> Agent间自主信息交换构成新经济形态")
print(f"5. 推理成本约束: 每次${REASONING_COST_PER_MATCH:.4f}, "
      f"累计中介交易{int(model_df['total_intermediary_transactions'].iloc[-1])}次")
print(f"   -> 推理成本是AI中介经济可行性的核心约束")


## 8. 天道推演 × 多Agent仿真

本仿真本质是**计算化的天道推演沙盘**：

| 天道推演能力 | 仿真对应 | 涌现产出 |
|-------------|---------|---------|
| 局势感知 | 初始Agent分布与参数 | 初始基尼/价格 |
| 因果链追踪 | Agent行为因果（购买->降价->竞争） | 价格动态 |
| 沙盘模拟（3层） | 100 tick推演 | 时间序列涌现 |
| 概率评估 | 多次运行不同seed | 结果分布 |
| 最优路径推荐 | 对比不同参数场景 | 策略选择 |

**核心洞察**：Agent经济仿真让天道推演从"意识中的沙盘"变为"可计算、可复现的沙盘"。

## 交付物
- [ ] 完成的 starter.ipynb（6个TODO全部填好）
- [ ] 4个子图的仿真结果可视化
- [ ] 一段300字分析：仿真涌现了什么经济现象？推理成本对AI中介的影响？
